# Advanced Retrieval with Multi-Query Generation

Welcome to the core of advanced retrieval techniques. In standard Retrieval-Augmented Generation (RAG), we typically take a single user query, embed it, and search the vector store once. While effective, this approach often fails when the user's intent is complex, ambiguous, or spans multiple related concepts—a common occurrence in real-world enterprise applications.

This notebook introduces the **MultiQueryRetriever**, an essential component for building robust RAG systems. Instead of relying on a single query, the MultiQueryRetriever leverages a sophisticated Language Model (LLM) to automatically generate several semantically diverse and complementary queries based on the original user input. These multiple queries are then used independently to retrieve chunks from the vector store. By aggregating results from these varied perspectives, we dramatically increase the chances of retrieving all necessary context, significantly boosting both the recall and precision of our system.

Mastering multi-query retrieval is critical for advanced development using frameworks like LangGraph. When building complex agentic workflows, the ability to gather comprehensive context from multiple angles—rather than just one best guess—is what separates a basic chatbot from an expert knowledge assistant. By implementing this pattern, you learn how to make your RAG pipelines resilient, reliable, and capable of handling the nuanced complexity inherent in specialized domain knowledge.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Understand Limitations:** Identify the limitations of single-query retrieval when dealing with complex or ambiguous user inputs.
*   **Implement MultiQueryRetriever:** Configure and utilize the `MultiQueryRetriever` component from LangChain to automatically generate multiple search queries.
*   **Enhance Context Retrieval:** Demonstrate how using multiple derived queries improves the breadth and depth of retrieved context compared to standard embedding lookups.
*   **Apply Advanced RAG Patterns:** Understand how multi-query generation serves as a foundational pattern for building highly robust, enterprise-grade RAG pipelines suitable for integration into agentic workflows (e.g., LangGraph).


### Setup and Imports

This cell imports necessary libraries for setting up the advanced RAG pipeline. It includes components for environment variable management (`dotenv`), document handling (`Document`), vector storage (`InMemoryVectorStore`), embedding/LLM models (OpenAI), specialized retrieval logic (`MultiQueryRetriever`), and text chunking (`RecursiveCharacterTextSplitter`).


In [23]:
from dotenv import load_dotenv
# Load environment variables from a .env file for API keys and settings.
from langchain_core.documents import Document
# Used to represent structured documents (text content, metadata).
from langchain_core.vectorstores import InMemoryVectorStore
# A simple in-memory vector store for demonstration purposes.
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# Imports the embedding model and chat model from OpenAI.
from langchain_classic.retrievers import MultiQueryRetriever
# The core component that generates multiple queries to improve retrieval accuracy.
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Used for splitting large documents into smaller, manageable chunks.


In [24]:
load_dotenv()

True

### Initialization of Core Components

This cell initializes the essential components for our RAG pipeline: the embedding model and the Large Language Model (LLM). `OpenAIEmbeddings` converts text into numerical vectors, while `ChatOpenAI` provides the conversational AI engine that processes queries and generates final answers.


In [25]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

llm = ChatOpenAI(model="gpt-5-mini", temperature=0.3)


### Document Loading and Preparation

This cell initializes a list of `Document` objects, simulating the loading of diverse knowledge sources (e.g., research papers, technical reports). Each document contains specialized text content (`page_content`) and associated metadata (like `topic`), which is crucial for advanced retrieval techniques like multi-query or topic filtering.


In [26]:
docs = [
    Document(
        page_content=(
            "Biotechnology companies are developing novel protein-based therapies that target specific "
            "disease pathways with unprecedented precision. Synthetic biology techniques allow scientists "
            "to engineer microorganisms that produce pharmaceutical compounds at industrial scale. "
            "Bioreactor technologies have dramatically reduced the cost of producing monoclonal antibodies, "
            "making treatments for autoimmune diseases and cancers more accessible. Microbiome research is "
            "revealing how manipulating gut bacteria can influence everything from mental health to "
            "metabolic disorders."
        ),
        metadata={"topic": "biotechnology"},
    ),
    Document(
        page_content=(
            "Zero-trust architecture has become the gold standard for enterprise network security, "
            "requiring continuous verification rather than relying on perimeter defenses. Machine learning "
            "models now detect anomalous network behavior in real time, reducing the window between "
            "intrusion and detection from months to minutes. Ransomware attacks on critical infrastructure "
            "have forced governments to establish mandatory incident reporting requirements for healthcare "
            "and energy sectors. Post-quantum cryptography standards are being finalized to protect "
            "sensitive data against future quantum computing threats."
        ),
        metadata={"topic": "cybersecurity"},
    ),
    Document(
        page_content=(
            "Brain-computer interfaces are enabling paralyzed patients to control prosthetic limbs and "
            "communicate using only their neural signals. Optogenetics allows researchers to activate or "
            "silence specific neuron populations with light, accelerating the understanding of neural "
            "circuit function and disease. Advanced neuroimaging techniques using fMRI and "
            "magnetoencephalography are mapping brain connectivity with millimeter precision, unlocking "
            "new treatments for depression and PTSD. Neurofeedback therapies are showing promise for "
            "cognitive rehabilitation following traumatic brain injuries."
        ),
        metadata={"topic": "neuroscience"},
    ),
    Document(
        page_content=(
            "Perovskite solar cells have achieved efficiency ratings exceeding 33%, surpassing traditional "
            "silicon panels and promising dramatically lower manufacturing costs. Grid-scale battery "
            "storage using iron-air and sodium-ion technologies is making renewable energy dispatchable "
            "around the clock without relying on rare earth metals. Offshore floating wind farms are "
            "expanding into deep-water regions previously inaccessible to fixed-foundation turbines, "
            "multiplying available wind energy capacity. Green hydrogen produced via electrolysis is "
            "emerging as a critical energy carrier for decarbonizing heavy industry and long-haul "
            "transport."
        ),
        metadata={"topic": "renewable_energy"},
    ),
    Document(
        page_content=(
            "Surgical robots equipped with haptic feedback allow surgeons to perform minimally invasive "
            "procedures with sub-millimeter precision, reducing patient recovery times significantly. "
            "Collaborative robots in manufacturing now work safely alongside humans using advanced "
            "computer vision and force sensing, without the need for physical barriers. Autonomous mobile "
            "robots are transforming warehouse logistics, optimizing pick-and-place operations and "
            "reducing fulfillment errors. Soft robots inspired by biological organisms are being developed "
            "for delicate tasks in agriculture, search-and-rescue, and medical drug delivery."
        ),
        metadata={"topic": "robotics"},
    ),
    Document(
        page_content=(
            "Base editing and prime editing technologies offer more precise alternatives to CRISPR-Cas9, "
            "enabling single-letter corrections to the genome without creating double-strand breaks. "
            "Gene therapy trials using adeno-associated virus vectors have achieved functional cures for "
            "hemophilia B and spinal muscular atrophy. Epigenome editing tools allow researchers to "
            "switch genes on or off without altering the underlying DNA sequence, opening new avenues "
            "for treating complex diseases. Polygenic risk scoring combined with germline analysis is "
            "enabling predictive medicine that identifies disease susceptibility decades before symptoms "
            "appear."
        ),
        metadata={"topic": "genetic_engineering"},
    ),
]

print(f"Created {len(docs)} documents")



Created 6 documents


### Document Chunking (Text Splitting)

This cell uses `RecursiveCharacterTextSplitter` to break down large documents into smaller, manageable chunks. This process is crucial for RAG because embedding models have token limits and smaller chunks improve the precision of retrieval by ensuring that retrieved context is highly focused.


In [27]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

docs = splitter.split_documents(docs)

print(f"Chunks: {len(docs)}")


Chunks: 18


input_query --> llm --> 3 queries (variants) --> retrieval --> 9 docs --> deduplication

### Vectorstore Initialization and Retriever Setup

This cell first initializes an in-memory vector store from the loaded documents (`docs`) using specified embeddings. It then converts this vector store into a retriever object, which is essential for querying the knowledge base by retrieving the top 3 most relevant chunks of information.


In [28]:
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### Code Explanation

This cell executes the retrieval step using a specialized retriever (`base_retriever`) to fetch documents relevant to the given query. It then iterates through the retrieved results, printing both the document's metadata (specifically the 'topic') and its content for inspection.


In [31]:
results_with_sim_search = base_retriever.invoke("How are modern technologies improving human health?")

# Print the total count of unique documents retrieved.
print(f"Retrieved {len(results_with_sim_search)} unique documents:\n")

# Iterate through each document found in the results list.
for i, doc in enumerate(results_with_sim_search):
    # Print a header showing the result number and its topic metadata.
    print(f"--- Result {i+1} [{doc.metadata.get('topic')}] ---")
    # Print the actual content of the retrieved document.
    print(doc.page_content)
    # Add an extra newline for clean separation between results.
    print()



Retrieved 3 unique documents:

--- Result 1 [biotechnology] ---
Biotechnology companies are developing novel protein-based therapies that target specific disease pathways with unprecedented precision. Synthetic biology techniques allow scientists to engineer microorganisms that produce pharmaceutical compounds at industrial scale. Bioreactor technologies have

--- Result 2 [robotics] ---
Surgical robots equipped with haptic feedback allow surgeons to perform minimally invasive procedures with sub-millimeter precision, reducing patient recovery times significantly. Collaborative robots in manufacturing now work safely alongside humans using advanced computer vision and force sensing,

--- Result 3 [biotechnology] ---
at industrial scale. Bioreactor technologies have dramatically reduced the cost of producing monoclonal antibodies, making treatments for autoimmune diseases and cancers more accessible. Microbiome research is revealing how manipulating gut bacteria can influence everything

### Multi-Query Retrieval Strategy

The `MultiQueryRetriever` automatically generates several alternative phrasings of the user's query using an LLM. It then executes retrieval for each phrasing and combines the results, significantly expanding recall without needing manual input from the user.


In [29]:
# MultiQueryRetriever generates multiple alternative phrasings of the user question,
# retrieves docs for each, and returns the deduplicated union — expanding recall
# without requiring the user to manually craft multiple queries

retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)


### Code Explanation

This cell executes the core retrieval step. It takes a user query and passes it to the `retriever` object (which is typically an instance of a vector store retriever) using the `.invoke()` method. This function fetches relevant documents from the knowledge base, which are then printed out for inspection.


In [30]:
query = "How are modern technologies improving human health?"

# Use the retriever to fetch documents based on the query.
results = retriever.invoke(query)

# Print a summary of the retrieved results.
print(f"Retrieved {len(results)} unique documents:\n")
for i, doc in enumerate(results):
    # Access metadata (like 'topic') for better context display.
    print(f"--- Result {i+1} [{doc.metadata.get('topic')}] ---")
    # Print the actual content of the retrieved document chunk.
    print(doc.page_content)
    print()


Retrieved 6 unique documents:

--- Result 1 [robotics] ---
Surgical robots equipped with haptic feedback allow surgeons to perform minimally invasive procedures with sub-millimeter precision, reducing patient recovery times significantly. Collaborative robots in manufacturing now work safely alongside humans using advanced computer vision and force sensing,

--- Result 2 [genetic_engineering] ---
combined with germline analysis is enabling predictive medicine that identifies disease susceptibility decades before symptoms appear.

--- Result 3 [biotechnology] ---
Biotechnology companies are developing novel protein-based therapies that target specific disease pathways with unprecedented precision. Synthetic biology techniques allow scientists to engineer microorganisms that produce pharmaceutical compounds at industrial scale. Bioreactor technologies have

--- Result 4 [genetic_engineering] ---
Base editing and prime editing technologies offer more precise alternatives to CRISPR-Cas9, e

### Multi-Query Retrieval Strategy

This cell initializes and uses a `MultiQueryRetriever`, which is an advanced technique that leverages an LLM to generate multiple, semantically diverse queries from the original user input. By using `include_original=True`, we ensure the original query is always included in the search set, maximizing the chances of finding relevant documents regardless of phrasing mismatch.


In [32]:
# include_original=True adds the original user query to the set of queries,
# ensuring docs that match the original phrasing are never missed

retriever_with_original = MultiQueryRetriever.from_llm(
    retriever=base_retriever, 
    llm=llm, 
    include_original=True
)

# Invoke the multi-query retriever with the user's query
results_with_original = retriever_with_original.invoke(query)

print(f"Retrieved {len(results_with_original)} unique documents (include_original=True):\n")
for i, doc in enumerate(results_with_original):
    # Print the result number and its associated topic metadata
    print(f"--- Result {i+1} [{doc.metadata.get('topic')}] ---")
    # Print the actual content of the retrieved document
    print(doc.page_content)
    print()



Retrieved 6 unique documents (include_original=True):

--- Result 1 [genetic_engineering] ---
combined with germline analysis is enabling predictive medicine that identifies disease susceptibility decades before symptoms appear.

--- Result 2 [biotechnology] ---
Biotechnology companies are developing novel protein-based therapies that target specific disease pathways with unprecedented precision. Synthetic biology techniques allow scientists to engineer microorganisms that produce pharmaceutical compounds at industrial scale. Bioreactor technologies have

--- Result 3 [neuroscience] ---
Brain-computer interfaces are enabling paralyzed patients to control prosthetic limbs and communicate using only their neural signals. Optogenetics allows researchers to activate or silence specific neuron populations with light, accelerating the understanding of neural circuit function and disease.

--- Result 4 [biotechnology] ---
at industrial scale. Bioreactor technologies have dramatically reduced 